# Fig. 3: post-mortem cortex, source tables

Single-nucleus RNA-seq of prefrontal (PFC) and anterior cingulate (ACC) cortex from Velmeshev et al. 2019 (41 samples from 31 individuals: 22 ASD and 19 control samples; 10 individuals contributed both regions). 3'UTR lengths were estimated from Read 2 positions and tested per gene and per cell type in R for the 2020 preprint of this work; the stored results are read here and written as tables for the panel notebook `fig3_cortex_panels`.

Inputs (the stored 2020 results in `copied_manuel_data/Mapping_3UTR_10X/` below the data root, or the same files written by `analysis/cortex/` of the code repository; set `M26_CORTEX_RESULTS` to that folder):
- `asd_utr_data.rda`: per-gene mean 3'UTR length for each sample and cell type (`mat_list`), P values of the per-gene linear model (`lm_list`) and the sample table (`lm_tbl`);
- `meta_utr_asd_.rda`: mean 3'UTR length over all 3'UTR molecules of each sample and cell type (`meta_utr_asd_tbl`) and the adjusted P values used in the 2020 preprint (`meta_utr_asd_test_tbl`);
- `asd_pathway.rda`: Reactome gene-set enrichment for layer 2/3 excitatory neurons (`path_l23_tbl`).

The figure uses the stored 2020 results; rerunning the enrichment with the current Reactome release gives more pathways (see `analysis/cortex/03-261012-gsea_l23.R`).

The input files are only read.

## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); tables are written to `M26_RESULTS_ROOT/Fig3/fig3_cortex_tables/`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
REPO_ROOT <- Sys.getenv("M26_REPO_ROOT")
if (!nzchar(REPO_ROOT)) {
  REPO_ROOT <- normalizePath(getwd())
  while (!file.exists(file.path(REPO_ROOT, "environment", "apa-python.yml")) && REPO_ROOT != dirname(REPO_ROOT)) {
    REPO_ROOT <- dirname(REPO_ROOT)
  }
  Sys.setenv(M26_REPO_ROOT = REPO_ROOT)
}
DATA_ROOT <- Sys.getenv("M26_DATA_ROOT")
if (!nzchar(DATA_ROOT)) stop("Set M26_DATA_ROOT (see README)")
RESULTS_ROOT <- Sys.getenv("M26_RESULTS_ROOT", file.path(REPO_ROOT, "results"))
OUT_DIR <- file.path(RESULTS_ROOT, "Fig3", "fig3_cortex_tables")

In [ ]:
suppressPackageStartupMessages({
  library(AnnotationDbi)
  library(dplyr)
  library(org.Hs.eg.db)
  library(tibble)
})
# results of the cortex analysis: the stored 2020 results by default, or the output folder of
# analysis/cortex/ in the code repository (set M26_CORTEX_RESULTS)
CORTEX_DIR <- Sys.getenv("M26_CORTEX_RESULTS", file.path(DATA_ROOT, "copied_manuel_data", "Mapping_3UTR_10X"))
dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

load(file.path(CORTEX_DIR, "asd_utr_data.rda")) # mat_list, lm_list, lm_tbl
load(file.path(CORTEX_DIR, "meta_utr_asd_.rda")) # meta_utr_asd_tbl, meta_utr_asd_test_tbl
load(file.path(CORTEX_DIR, "asd_pathway.rda")) # path_l23_tbl, path_astpp_tbl

write_tsv <- function(df, name) {
  write.table(df, file.path(OUT_DIR, name), sep = "\t", quote = FALSE, row.names = FALSE, na = "")
}

## Samples

One row per sample (individual and region). The rows of `lm_tbl` are the rows of every matrix in `mat_list`.

In [ ]:
samples <- lm_tbl |>
  transmute(
    individual = ID,
    region = ifelse(grepl("prefrontal", Region), "PFC", "ACC"),
    diagnosis = Diagnose,
    sex = Sex,
    age = Age
  )
stopifnot(all(sapply(mat_list, nrow) == nrow(samples)))
print(table(samples$diagnosis, samples$region))
cat("individuals:", length(unique(samples$individual)), "\n")
write_tsv(samples, "fig3_samples.tsv")

## Fig. 3B: mean 3'UTR length per sample and cell type

Each value is the mean over all 3'UTR molecules detected in the nuclei of one sample and cell type. ASD and control samples are compared with a two-sided Welch t-test per cell type, and the P values are adjusted with the Benjamini-Hochberg procedure across the 17 cell types. The 2020 preprint adjusted the same P values jointly with all other pairwise comparisons between the 34 cell-type and diagnosis groups (`pairwise.t.test`); those values are kept in the column `fdr_preprint` for comparison.

In [ ]:
meta <- meta_utr_asd_tbl |>
  transmute(
    individual = as.character(Individual),
    region = Region,
    diagnosis = as.character(Diagnose),
    cell_type = as.character(Cell_Type),
    utr3_length = UTR3_Length
  )

tests <- meta |>
  group_by(cell_type) |>
  summarise(
    n_asd = sum(diagnosis == "ASD" & !is.na(utr3_length)),
    n_control = sum(diagnosis == "Control" & !is.na(utr3_length)),
    mean_asd = mean(utr3_length[diagnosis == "ASD"], na.rm = TRUE),
    mean_control = mean(utr3_length[diagnosis == "Control"], na.rm = TRUE),
    p = t.test(utr3_length[diagnosis == "ASD"], utr3_length[diagnosis == "Control"])$p.value,
    .groups = "drop"
  ) |>
  mutate(
    difference = mean_asd - mean_control,
    fdr = p.adjust(p, method = "BH"),
    fdr_preprint = if (exists("meta_utr_asd_test_tbl")) {
      meta_utr_asd_test_tbl$p.adjust[match(cell_type, meta_utr_asd_test_tbl$Cell_Type)]
    } else {
      NA_real_
    }
  ) |>
  arrange(desc(difference))

# cell-type order of the panel (top to bottom), as in the 2020 preprint
ct_order <- rev(levels(meta_utr_asd_tbl$Cell_Type))
tests$panel_order <- match(tests$cell_type, ct_order)
print(as.data.frame(tests), digits = 3)
cat("FDR < 5%:", sum(tests$fdr < 0.05), " FDR < 10%:", sum(tests$fdr < 0.1), "of", nrow(tests), "\n")
write_tsv(meta, "fig3b_sample_celltype_length.tsv")
write_tsv(tests, "fig3b_celltype_tests.tsv")

## Per-gene tests

For every gene and cell type, the per-sample mean 3'UTR length was modelled as `length ~ Sex + Region + Diagnosis` and compared with `length ~ Sex + Region` by a likelihood-ratio test (`anova(..., test = "LRT")`), for genes with a value in more than 10 samples. Columns of `mat_list` are Entrez gene identifiers; a suffix `_k` marks the k-th separate 3'UTR region of a gene. The P values are adjusted with the Benjamini-Hochberg procedure within each cell type.

In [ ]:
gene_tbl <- function(ct) {
  m <- mat_list[[ct]]
  p <- lm_list[[ct]]
  asd <- samples$diagnosis == "ASD"
  mean_asd <- colMeans(m[asd, , drop = FALSE], na.rm = TRUE)
  mean_control <- colMeans(m[!asd, , drop = FALSE], na.rm = TRUE)
  tibble(
    cell_type = ct,
    utr_id = colnames(m),
    entrez = sub("_.*", "", colnames(m)),
    n_samples = colSums(!is.na(m)),
    mean_asd = mean_asd,
    mean_control = mean_control,
    difference = mean_asd - mean_control,
    log_ratio = log(mean_asd / mean_control),
    p = unname(p[colnames(m)]),
    fdr = p.adjust(p, method = "BH")
  )
}
genes <- bind_rows(lapply(names(mat_list), gene_tbl))
genes$symbol <- unname(mapIds(
  org.Hs.eg.db, unique(genes$entrez), "SYMBOL", "ENTREZID",
  multiVals = "first"
)[genes$entrez])

summary_tbl <- genes |>
  group_by(cell_type) |>
  summarise(
    tested = sum(!is.na(p)),
    p_below_0.05 = sum(p < 0.05, na.rm = TRUE),
    longer_p_below_0.05 = sum(p < 0.05 & difference > 0, na.rm = TRUE),
    shorter_p_below_0.05 = sum(p < 0.05 & difference < 0, na.rm = TRUE),
    fdr_below_0.05 = sum(fdr < 0.05, na.rm = TRUE),
    fdr_below_0.10 = sum(fdr < 0.1, na.rm = TRUE),
    .groups = "drop"
  )
print(as.data.frame(summary_tbl))
write_tsv(genes, "fig3_gene_tests_all_cell_types.tsv")
write_tsv(summary_tbl, "fig3_gene_tests_summary.tsv")

## Fig. 3C and 3D: layer 2/3 excitatory neurons

Genes of the volcano plot (tested genes only) and the per-sample lengths of SOX4 and NCAM2.

In [ ]:
l23 <- genes |> filter(cell_type == "L2/3", !is.na(p))
print(as.data.frame(l23 |> filter(fdr < 0.1) |> arrange(p)), digits = 3)
print(as.data.frame(l23 |> filter(symbol %in% c("SOX4", "NCAM2", "BDNF")) |> arrange(p)), digits = 3)
write_tsv(l23, "fig3c_l23_genes.tsv")

examples <- c(SOX4 = "6659", NCAM2 = "4685")
d <- bind_rows(lapply(names(examples), function(g) {
  samples |> mutate(gene = g, utr3_length = mat_list[["L2/3"]][, examples[[g]]])
}))
write_tsv(d, "fig3d_l23_examples.tsv")

## Fig. 3E: Reactome gene-set enrichment, layer 2/3 excitatory neurons

3'UTR regions were ranked by the log ratio of their mean length (ASD over control); a gene with several regions enters the pathways through its highest-ranked region. Simple permutation test with 10,000 permutations; pathways with an adjusted P below 0.1 were kept in `path_l23_tbl`.

In [ ]:
gsea <- as_tibble(path_l23_tbl) |>
  mutate(Description = as.character(Description)) |>
  arrange(desc(NES))
print(as.data.frame(gsea[, c("ID", "Description", "setSize", "NES", "pvalue", "p.adjust")]), digits = 3)
write_tsv(gsea, "fig3e_l23_reactome_gsea.tsv")

## Session

In [ ]:
sessionInfo()